# MemoryShards — face-embedding fine-tuning

Trains the one component of the MemoryShards pipeline that's actually learned, not just downloaded: a face-embedding model fine-tuned on a small **CelebA** subset (public, licensed, labeled identities), so it can later group *our own* photos by person the way a phone gallery does — via embedding similarity, not per-person classification.

**Infra note:** this notebook is meant to run on Colab's free GPU. It mounts Google Drive so checkpoints survive a session timeout/disconnect — re-running the notebook resumes from the last saved epoch instead of starting over.

**Honesty note:** this trains the model. It does not touch our own personal photos — those only get run through the *finished* model later, and any face-clustering output on personal photos stays out of anything published (same rule as no team names in the public repo).

## 1. Setup

In [ ]:
!pip install -q --no-deps facenet-pytorch datasets


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Runtime > Change runtime type > GPU)')


## 2. Mount Drive (checkpoint storage)

Checkpoints go to `MyDrive/MemoryShards/checkpoints/` so a disconnected Colab session doesn't lose progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/MemoryShards/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'face_embedding_head.pt')
print('checkpoints will be saved to', CHECKPOINT_PATH)


## 3. Download a small CelebA subset

Same source and logic as `src/download_celeba_subset.py` in the repo — the `flwrlabs/celeba` mirror on the Hugging Face Hub, which serves the official CelebA images plus `celeb_id` identity labels with no Google Drive/Kaggle auth needed. Rows are grouped by identity, so we stream just the first N identity blocks instead of the full ~200K images.

In [ ]:
TARGET_IDENTITIES = 120
MAX_IMAGES_PER_IDENTITY = 20
MIN_IMAGES_PER_IDENTITY = 10
MAX_ROWS_TO_SCAN = 20000
DATA_DIR = '/content/celeba_subset'


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path
from datasets import load_dataset

out_dir = Path(DATA_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ds = load_dataset('flwrlabs/celeba', split='train', streaming=True)

per_identity_count = defaultdict(int)
manifest_rows = []
identities_seen = []
rows_scanned = 0

for example in ds:
    rows_scanned += 1
    if rows_scanned > MAX_ROWS_TO_SCAN:
        break
    cid = example['celeb_id']
    if cid not in identities_seen:
        if len(identities_seen) >= TARGET_IDENTITIES:
            break
        identities_seen.append(cid)
    if per_identity_count[cid] >= MAX_IMAGES_PER_IDENTITY:
        continue
    idx = per_identity_count[cid]
    person_dir = out_dir / str(cid)
    person_dir.mkdir(exist_ok=True)
    img_path = person_dir / f'{idx}.jpg'
    example['image'].convert('RGB').save(img_path, 'JPEG', quality=92)
    manifest_rows.append((img_path.relative_to(out_dir).as_posix(), cid))
    per_identity_count[cid] += 1

kept_rows = [r for r in manifest_rows if per_identity_count[r[1]] >= MIN_IMAGES_PER_IDENTITY]
kept_identities = sorted({cid for _, cid in kept_rows})
print(f'scanned {rows_scanned} rows -> kept {len(kept_rows)} images across {len(kept_identities)} identities')


## 4. Pair dataset + online hard-negative mining

**Finding from local smoke-testing before this ever touched Colab:** the pretrained VGGFace2 backbone already separates these identities by a wide margin. Naive random triplets (random anchor/positive/negative) satisfy a margin=0.2 triplet loss immediately — every loss came out `0.0000`, meaning zero gradient signal, before a single real epoch. That's not a training bug, it's "the triplets were too easy."

Fix: the dataset only returns **(anchor, positive, identity)** pairs — no explicit negative. Each training step mines the *hardest* (closest) different-identity embedding **from within the same batch** as the negative. Verified locally: this turns loss from a flat `0.0000` into real, decreasing values with actual gradient signal.

In [ ]:
import random
from collections import defaultdict as _dd
from PIL import Image
from torch.utils.data import Dataset
from facenet_pytorch import MTCNN, fixed_image_standardization
import torchvision.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=14, device=device, post_process=False)

images_by_identity = _dd(list)
for path, cid in kept_rows:
    images_by_identity[cid].append(str(out_dir / path))
identity_list = list(images_by_identity.keys())
id_to_idx = {cid: i for i, cid in enumerate(identity_list)}


def load_face_tensor(path):
    img = Image.open(path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        # detector failed to find a face — fall back to a plain resize
        # so a handful of hard images don't crash a batch
        face = T.functional.resize(T.functional.to_tensor(img) * 255, [160, 160])
        face = fixed_image_standardization(face)
    return face


class CelebAPairDataset(Dataset):
    """Returns (anchor, positive, identity_index). No explicit negative —
    negatives are mined inside the training loop from the rest of the batch."""

    def __init__(self, images_by_identity, identity_list, id_to_idx, length=2000):
        self.images_by_identity = images_by_identity
        self.identity_list = identity_list
        self.id_to_idx = id_to_idx
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, _):
        identity = random.choice(self.identity_list)
        anchor_path, positive_path = random.sample(self.images_by_identity[identity], 2)
        return (
            load_face_tensor(anchor_path),
            load_face_tensor(positive_path),
            self.id_to_idx[identity],
        )


def batch_hard_triplet_loss(anchor_emb, positive_emb, labels, margin=0.2):
    """For each anchor, mine the hardest (closest) negative from a
    different identity within the same batch — anchors+positives pooled
    together as the negative candidate pool."""
    all_emb = torch.cat([anchor_emb, positive_emb], dim=0)
    all_labels = torch.cat([labels, labels], dim=0)

    dist_matrix = torch.cdist(anchor_emb, all_emb)  # (B, 2B)
    B = anchor_emb.size(0)

    negative_emb = []
    for i in range(B):
        different_identity = all_labels != labels[i]
        candidate_dists = dist_matrix[i].clone()
        candidate_dists[~different_identity] = float('inf')
        hardest_idx = torch.argmin(candidate_dists)
        negative_emb.append(all_emb[hardest_idx])
    negative_emb = torch.stack(negative_emb)

    d_ap = (anchor_emb - positive_emb).norm(dim=1)
    d_an = (anchor_emb - negative_emb).norm(dim=1)
    per_sample = torch.clamp(d_ap - d_an + margin, min=0.0)
    return per_sample, d_ap, d_an


train_dataset = CelebAPairDataset(images_by_identity, identity_list, id_to_idx, length=2000)
print('pair dataset ready —', len(identity_list), 'identities, negatives mined per-batch')


## 5. Model — pretrained backbone, fine-tune only the head

Matches the plan in `context/PROJECT.md`: freeze everything except the last linear + batchnorm layer.

**Note on `num_workers=0`:** MTCNN runs on `cuda` inside the dataset — spawning DataLoader worker subprocesses (`num_workers>0`) forks a process that already has a CUDA context initialized, which crashes with `Cannot re-initialize CUDA in forked subprocess`. The subset is small enough that single-process loading isn't a bottleneck, so we just keep it simple instead of switching multiprocessing start methods.

In [ ]:
from torch import nn, optim
from torch.utils.data import DataLoader
from facenet_pytorch import InceptionResnetV1

model = InceptionResnetV1(pretrained='vggface2', classify=False).to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.last_linear.parameters():
    param.requires_grad = True
for param in model.last_bn.parameters():
    param.requires_grad = True

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f'resumed from checkpoint at epoch {start_epoch}')
else:
    print('no checkpoint found — starting fresh')

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)


## 6. Train

A handful of epochs over the small subset — this is fine-tuning a head, not training from scratch. Saves a checkpoint after every epoch, and logs how many triplets in each batch actually produced a nonzero loss (a sanity signal — if this drops to 0/32 every batch, the remaining triplets have all become easy and more epochs won't help).

In [ ]:
NUM_EPOCHS = 8

model.train()
for epoch in range(start_epoch, NUM_EPOCHS):
    running_loss = 0.0
    running_nonzero = 0
    running_total = 0
    for anchors, positives, label_idx in train_loader:
        anchors = anchors.to(device)
        positives = positives.to(device)
        label_idx = label_idx.to(device)

        emb_a = model(anchors)
        emb_p = model(positives)
        per_sample_loss, d_ap, d_an = batch_hard_triplet_loss(emb_a, emb_p, label_idx)
        loss = per_sample_loss.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_nonzero += (per_sample_loss > 0).sum().item()
        running_total += len(per_sample_loss)

    avg_loss = running_loss / len(train_loader)
    print(f'epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {avg_loss:.4f}  '
          f'nonzero triplets: {running_nonzero}/{running_total}')

    torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch}, CHECKPOINT_PATH)


## 7. Sanity check

Same-identity pairs should score higher cosine similarity than different-identity pairs. This isn't a rigorous eval — just confirms the fine-tuning moved the embeddings in the right direction.

In [ ]:
import torch.nn.functional as F

model.eval()
with torch.no_grad():
    id_a, id_b = random.sample(identity_list, 2)
    same_1, same_2 = random.sample(images_by_identity[id_a], 2)
    diff_1 = random.choice(images_by_identity[id_a])
    diff_2 = random.choice(images_by_identity[id_b])

    e_same_1 = model(load_face_tensor(same_1).unsqueeze(0).to(device))
    e_same_2 = model(load_face_tensor(same_2).unsqueeze(0).to(device))
    e_diff_1 = model(load_face_tensor(diff_1).unsqueeze(0).to(device))
    e_diff_2 = model(load_face_tensor(diff_2).unsqueeze(0).to(device))

    sim_same = F.cosine_similarity(e_same_1, e_same_2).item()
    sim_diff = F.cosine_similarity(e_diff_1, e_diff_2).item()

    print(f'same-person similarity:      {sim_same:.3f}')
    print(f'different-person similarity: {sim_diff:.3f}')
    print('PASS: same > different' if sim_same > sim_diff else 'WARNING: same <= different — needs more epochs/data')


## 8. Next

- Checkpoint is saved to Drive at `MemoryShards/checkpoints/face_embedding_head.pt` — download it into the repo's `checkpoints/` folder (gitignored) for local inference.
- Day 8 of the plan: run this trained model on our own photos (MTCNN detect -> embed -> cluster with DBSCAN/agglomerative on cosine distance) to group photos by person.
- Reminder: don't publish face-clustering output on personal photos anywhere public.